# Importations alimentaires des Comores

Notebook du projet FAOSTAT. Les données brutes sont conservées sans modification.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
FIGURES_DIR = ROOT_DIR / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Diagnostic de la série temporelle

In [2]:
FILE = PROCESSED_DIR / "imports_comoros_processed.csv"
if not FILE.exists(): raise FileNotFoundError("Exécute le notebook 01.")
data = pd.read_csv(FILE)
data["Year"] = pd.to_numeric(data["Year"], errors="coerce")
yearly = data.groupby("Year", as_index=False).agg(
    quantity_t=("Quantity (t)", "sum"),
    value_1000_usd=("Value (1000 USD)", "sum")
).dropna(subset=["Year"]).sort_values("Year")
yearly["gap"] = yearly["Year"].diff()
display(yearly)

,Year,quantity_t,value_1000_usd,gap
0,1995,"97,512.05","43,756.00",NaN
1,1996,"73,206.25","32,446.00",1.00
2,1997,"80,876.20","34,181.00",1.00
3,1998,"39,982.00","27,401.00",1.00
4,1999,"41,009.70","18,460.00",1.00
5,2000,"58,779.00","20,747.00",1.00
6,2001,"43,582.10","16,640.00",1.00
7,2002,"55,035.00","21,548.00",1.00
8,2016,"245,906.92","108,183.00",14.00
9,2017,"184,630.49","112,663.00",1.00


## 2. Extraire la plus longue séquence d'années consécutives

In [3]:
def longest_consecutive_segment(frame):
    frame = frame.dropna(subset=["Year"]).sort_values("Year").copy()
    if frame.empty: return frame
    years = frame["Year"].astype(int).to_numpy()
    starts = [0]
    ends = []
    for i in range(1, len(years)):
        if years[i] != years[i-1] + 1:
            ends.append(i); starts.append(i)
    ends.append(len(years))
    segments = [frame.iloc[s:e] for s,e in zip(starts,ends)]
    return max(segments, key=len).copy()

segment = longest_consecutive_segment(yearly)
display(segment)
print("Nombre d'années consécutives :", len(segment))

,Year,quantity_t,value_1000_usd,gap
0,1995,"97,512.05","43,756.00",NaN
1,1996,"73,206.25","32,446.00",1.00
2,1997,"80,876.20","34,181.00",1.00
3,1998,"39,982.00","27,401.00",1.00
4,1999,"41,009.70","18,460.00",1.00
5,2000,"58,779.00","20,747.00",1.00
6,2001,"43,582.10","16,640.00",1.00
7,2002,"55,035.00","21,548.00",1.00


Nombre d'années consécutives : 8


## 3. Modèle linéaire de référence

Le modèle est exécuté uniquement si la plus longue séquence continue contient au moins 8 observations. Le test respecte l'ordre chronologique.

In [4]:
if len(segment) < 8:
    print("Série continue trop courte : pas de prévision fiable avec ce protocole.")
else:
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    target = "value_1000_usd"
    model_data = segment.dropna(subset=[target]).copy()
    n_test = max(2, int(np.ceil(len(model_data)*0.25)))
    train, test = model_data.iloc[:-n_test], model_data.iloc[-n_test:]

    model = LinearRegression()
    model.fit(train[["Year"]], train[target])
    pred = model.predict(test[["Year"]])

    print("MAE :", mean_absolute_error(test[target], pred))
    print("RMSE :", mean_squared_error(test[target], pred)**0.5)
    print("R² :", r2_score(test[target], pred))
    result = test[["Year", target]].copy()
    result["prediction"] = pred
    display(result)

MAE : 8313.55714285653
RMSE : 9596.631696185885
R² : -14.292853859380916


,Year,value_1000_usd,prediction
6,2001,"16,640.00","13,120.20"
7,2002,"21,548.00","8,440.69"
